In [ ]:
import polars as pl
df = pl.scan_parquet("../data/processed/parquet/20250101.parquet").collect()

In [31]:
df.head()

Fecha_Transaccion,Tiempo,Estacion,Linea,Acceso_Estacion,Entradas_E,Salidas_S,timestamp
str,str,str,str,str,i64,i64,datetime[μs]
"""2025-01-01""","""00:00:00""","""(02000)Cabecera Autopista Nort…","""(33)Zona B AutoNorte""","""(KB) BATERIA PLATAFORMA Orient…",0,17,2025-01-01 00:00:00
"""2025-01-01""","""00:00:00""","""(02001)Centro Comercial Santa …","""(33)Zona B AutoNorte""","""(01) Acceso Norte""",0,1,2025-01-01 00:00:00
"""2025-01-01""","""00:00:00""","""(02101)Toberin - Foundever""","""(33)Zona B AutoNorte""","""(01) Acceso Norte""",0,3,2025-01-01 00:00:00
"""2025-01-01""","""00:00:00""","""(02102)Calle 161""","""(33)Zona B AutoNorte""","""(01) Acceso Norte""",0,1,2025-01-01 00:00:00
"""2025-01-01""","""00:00:00""","""(02103)Mazurén""","""(33)Zona B AutoNorte""","""(01) Acceso Norte""",0,1,2025-01-01 00:00:00


In [ ]:
df = (
    df
    # Delete unnecessary columns
    .drop("Fecha_Transaccion", "Tiempo", "Acceso_Estacion")
    # Rename columns
    .rename({
        "Entradas_E":"entradas",
        "Salidas_S":"salidas"
    })
    # Columns transformations
    .with_columns(
        # Derivate the hour and date column for the "timestamp" variable
        pl.col("timestamp").dt.date().alias("fecha"),
        pl.col("timestamp").dt.time().alias("hora"),
        # Extract two columns, one for the station´s code and other for the name
        pl.col("Estacion").str.extract(r"\((\d+)\)", group_index=1).alias("codigo").cast(pl.Int64),
        pl.col("Estacion").str.extract(r"\)(.+)", group_index=1).alias("nombre"),
        # Create the total validation per time column
        (pl.col("entradas") + pl.col("salidas")).alias("total")
    )
    # Delete unnecessary columns
    .drop("timestamp", "Estacion")
    # Sort the columns order
    .select(["fecha", "hora", "codigo", "nombre", "entradas", "salidas", "total"])
)
df.head()

fecha,hora,codigo,nombre,entradas,salidas,total
date,time,i64,str,i64,i64,i64
2025-01-01,00:00:00,2000,"""Cabecera Autopista Norte""",0,17,17
2025-01-01,00:00:00,2001,"""Centro Comercial Santa Fe""",0,1,1
2025-01-01,00:00:00,2101,"""Toberin - Foundever""",0,3,3
2025-01-01,00:00:00,2102,"""Calle 161""",0,1,1
2025-01-01,00:00:00,2103,"""Mazurén""",0,1,1


On this short Jupyter Notebook I´m just wanna looking the data cleaning made on the `dataset.py` file and aggregate some extras process. This transformations are already up on this file